# Fine-tuned Baselines: BERT, HateBERT, and RoBERTa on IHC and ISHate

We fine-tune three pretrained models on two hate speech corpora (IHC and ISHate) and compare their macro F1 scores across both datasets.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
import warnings
warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path("..").resolve()))

## 1. Load the Datasets

### IHC (Implicit Hate Corpus)
`SALT-NLP/implicit-hate` was removed from the Hub — `tasksource/implicit-hate-stg1` is a verified mirror of the same IHC corpus (ElSherief et al., 2021). It has a single `train` split with 21,480 tweets labelled as `not_hate`, `explicit_hate`, or `implicit_hate`. Since there is no pre-made test split, we hold out 10% for evaluation. The binary label collapses explicit and implicit hate into a single **HS** class (label = 1); non-hate is **Non-HS** (label = 0).

### ISHate (Implicit and Subtle Hate Speech)
`BenjaminOcampo/ISHate` is a large-scale dataset (63,758 examples) focused on implicit and subtle hate speech, annotated with multiple layers. We use the binary hate label for classification (`Non-HS` → 0, `HS` → 1) and its pre-made train/test splits.

In [ ]:
from data_loaders import load_ihc_binary, load_ishate_binary

train_ds, test_ds       = load_ihc_binary(seed=42)
ishate_train, ishate_test = load_ishate_binary()

print("IHC")
print(f"  Train: {len(train_ds):,}  Test: {len(test_ds):,}")
print("\nISHate")
print(f"  Train: {len(ishate_train):,}  Test: {len(ishate_test):,}")

### Vicomtech (Hate Speech Dataset — Stormfront)
`Vicomtech/hate-speech-dataset` (de Gibert et al., 2018) is a collection of sentences extracted from Stormfront white-nationalist forum posts. Each sentence is labelled via its metadata file. We use the pre-made train and test splits (balanced classes). The `idk/skip` and `relation` labels are discarded. The binary label maps `hate` → 1, `noHate` → 0.

In [ ]:
from data_loaders import load_vicomtech

vicomtech_train, vicomtech_test = load_vicomtech()

print("Vicomtech")
print(f"  Train: {len(vicomtech_train):,}  Test: {len(vicomtech_test):,}")

## 2. Tokenization

Transformer models cannot work directly on raw text — they require a **tokenizer** that converts text into integer token IDs and an attention mask.

We truncate and pad every sequence to a fixed length of 128 tokens. Most tweets are well under this limit, so little information is lost. Each model has its own vocabulary and tokenizer, so we re-tokenize for each model.

In [3]:
def tokenize(ds, tokenizer, text_col="post", max_length=128):
    def _tok(batch):
        return tokenizer(
            batch[text_col],
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )
    return ds.map(_tok, batched=True)

## 3. Metrics

The `Trainer` calls `compute_metrics` after each evaluation epoch. We use **macro F1** as the primary metric — it averages F1 across both classes, penalising models that ignore the minority class (hate speech). Precision and recall give additional diagnostic information.

In [ ]:
from training_utils import compute_metrics

## 4. Fine-tuning Loop

### What is a classification head?

A pretrained model like BERT has learned rich representations of text but has no built-in notion of hate speech. `AutoModelForSequenceClassification` adds a small **linear layer** (the classification head) on top of the `[CLS]` token representation, projecting it to 2 output logits (Non-HS, HS). This head is randomly initialised.

### What does fine-tuning do?

Fine-tuning runs gradient descent on the labelled IHC training set, updating **all** model weights: both the pretrained transformer layers and the new classification head. The pretrained weights are a warm start: the model has already learned grammar, semantics, and world knowledge, so it only needs a few epochs to adapt to the new task. Training from scratch would require far more data and compute.

In [ ]:
MODELS = {
    "bert":     "bert-base-uncased",
    "hatebert": "GroNLP/hateBERT",
    "roberta":  "roberta-base",
}

DATASETS = {
    "IHC":       {"train": train_ds,        "test": test_ds,        "text_col": "post"},
    "ISHate":    {"train": ishate_train,     "test": ishate_test,    "text_col": "text"},
    "Vicomtech": {"train": vicomtech_train,  "test": vicomtech_test, "text_col": "text"},
}

results = {}

for dataset_name, dataset in DATASETS.items():
    results[dataset_name] = {}
    for model_name, model_id in MODELS.items():
        print(f"\n{'='*60}")
        print(f"Dataset: {dataset_name}  |  Model: {model_name}")
        print(f"{'='*60}")

        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model     = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

        tok_train = tokenize(dataset["train"], tokenizer, text_col=dataset["text_col"])
        tok_test  = tokenize(dataset["test"],  tokenizer, text_col=dataset["text_col"])

        training_args = TrainingArguments(
            output_dir=f"../../checkpoints_baseline/{dataset_name}/{model_name}",
            num_train_epochs=3,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=32,
            learning_rate=2e-5,
            eval_strategy="epoch",
            save_strategy="no",
            logging_strategy="epoch",
            report_to="none",
            seed=42,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tok_train,
            eval_dataset=tok_test,
            compute_metrics=compute_metrics,
        )

        trainer.train()

        # Save fine-tuned weights in HuggingFace format
        save_path = f"../../weigths/weights_baseline/{model_name}/{dataset_name}"
        os.makedirs(save_path, exist_ok=True)
        trainer.save_model(save_path)
        tokenizer.save_pretrained(save_path)
        print(f"  Saved weights → {save_path}")

        preds_output = trainer.predict(tok_test)
        preds  = np.argmax(preds_output.predictions, axis=-1)
        labels = dataset["test"]["label"]

        print(classification_report(labels, preds, target_names=["Non-HS", "HS"]))

        results[dataset_name][model_name] = {
            "macro_f1":  f1_score(labels, preds, average="macro",  zero_division=0),
            "macro_p":   precision_score(labels, preds, average="macro", zero_division=0),
            "macro_r":   recall_score(labels, preds, average="macro",    zero_division=0),
        }

## 5. Results

One table per dataset shows macro F1/P/R for all three models. Comparing across datasets reveals whether model rankings are consistent or dataset-dependent — e.g., whether HateBERT's domain-specific pretraining helps more on ISHate's subtler examples than on IHC.

In [8]:
import pandas as pd

for dataset_name, dataset_results in results.items():
    df = pd.DataFrame(dataset_results).T
    df.columns = ["Macro F1", "Macro Precision", "Macro Recall"]
    df.index.name = "Model"
    styled = df.style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold; background-color: #d4f1d4").set_caption(dataset_name)
    display(styled)

ValueError: Length mismatch: Expected axis has 0 elements, new values have 3 elements